In [ ]:
import os
import pandas as pd
import psycopg2
from google import genai
from pgvector.psycopg2 import register_vector
import numpy as np

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [ ]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

In [ ]:
resume_df = get_resume('b6e8165a-b1af-4117-8a29-4a3a5fc95f32')
candidate_industries = resume_df["industries"][0]

matching_jobs_df = filter_job_postings(candidate_industries)
matching_jobs_ids = matching_jobs_df["id"].to_list()

market_hard_skills_df = get_position_skills(matching_jobs_ids, HARD_SKILLS_TABLE)
market_soft_skills_df = get_position_skills(matching_jobs_ids, SOFT_SKILLS_TABLE)

market_soft_skills_df = market_soft_skills_df.sort_values("job_id")
market_soft_skills_df["index"] = market_soft_skills_df.groupby("job_id").ngroup()

market_hard_skills_df = market_hard_skills_df.sort_values("job_id")
market_hard_skills_df["index"] = market_hard_skills_df.groupby("job_id").ngroup()

### String Matrixes

In [ ]:
string_hard_skills_df = market_hard_skills_df[["index", "skill_description"]]
string_soft_skills_df = market_soft_skills_df[["index", "skill_description"]]

hard_skills_size = string_hard_skills_df.groupby('index').size().max()
soft_skills_size = string_soft_skills_df.groupby('index').size().max()

string_hard_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in string_hard_skills_df.groupby('index')
])

string_soft_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in string_soft_skills_df.groupby('index')
])

### Vector Matrixes

In [ ]:
vector_hard_skills_df = market_hard_skills_df[["index", "embedding"]]
vector_soft_skills_df = market_soft_skills_df[["index", "embedding"]]

vector_hard_skills_matrix = np.array([
    np.pad(group['embedding'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in vector_hard_skills_df.groupby('index')
])

vector_soft_skills_matrix = np.array([
    np.pad(group['embedding'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in vector_soft_skills_df.groupby('index')
])
